# Build DF40

Crops DF40 ([YZY-stack/DF40](https://github.com/YZY-stack/DF40), CC BY-NC 4.0, non-commercial research only) with the lab's pipeline into a private dataset (`sources/df40.py`):

- the FF++-domain frames of the 31 known methods, for FF++ training and validation sequences only, one frame in `EVERY_FAKE`;
- the authors' crops of the FF++ real frames, every split (the test sequences are e4e's reals);
- the 9 unknown methods with the reals each ships, and e4e's inversions of FF++ test sequences.

The authors' frames are already tight face crops; RetinaFace finds the face again and the crop is cut at the lab's margin, so DF40 shares the other sources' framing. Where the authors' crop is tighter than that margin the lab crop is padded with black; the check below measures how much black is left inside the 1.3x framing the detectors are trained on.

The download links arrive by email after the access form and live in the Colab secret `DF40_LINKS`, a JSON object with the Drive ids `train_folder` (the folder the email calls "testing data", which holds the FF++-domain training frames), `test_folder` (the one it calls "training data", which holds the unknown methods) and `ffpp_real` (the FF++ real frames). Archives used in small part (the known methods, one frame in `EVERY_FAKE`) are read in place with HTTP byte ranges (`http_range_reader`, `remote_zip_members`): only the central directory and the kept members are fetched. Archives used almost entirely (the FF++ reals and the unknown methods) are downloaded in 64 MB ranges (`download_ranges`). Neither asks Drive for a whole download: in October 2026 whole downloads of several DF40 archives hit Drive's daily quota ("Quota exceeded"), and after that Drive refused ranges of the same files too, for about a day. Tens of thousands of small range requests from one address also make Drive refuse ranges for a while.

A rerun skips archives already cropped; their markers live in `BUILD_DIR`, so a backup of `BUILD_DIR` to the private `WORK_REPO` (cell "Backup") lets a new runtime continue where the last one stopped.

**Runtime:** Colab Pro, L4 or A100, about two hours. **Needs** the Colab secrets `HF_TOKEN` (write access) and `DF40_LINKS`.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-df40"  # private
BUILD_DIR = "/content/build_df40"
ZIP_DIR = "/content/df40_zips"  # archives while they are cropped
WORK_REPO = "paulohenriquevn/face-decision-df40-work"  # private backup of BUILD_DIR, to resume on a new runtime
EVERY_FAKE = 16  # one frame in 16 of each known method: about 2 per fake video

In [ ]:
%%capture
!pip install -q "face-decision-lab[build] @ {REPO_SOURCE}" gdown
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu

In [ ]:
import os
os.environ["DEEPFACE_BACKEND_ENGINE"] = "onnx"

import onnxruntime as ort
providers = ort.get_available_providers()
print("ONNX Runtime providers:", providers)
assert "CUDAExecutionProvider" in providers, "onnxruntime-gpu cannot see the GPU; RetinaFace would run on the CPU"

### Archives

In [ ]:
import json, re, html
from pathlib import Path
import requests
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))
links = json.loads(userdata.get("DF40_LINKS"))

def folder_files(folder_id):
    page = requests.get(f"https://drive.google.com/embeddedfolderview?id={folder_id}", timeout = 60).text
    entries = re.findall(r'file/d/([^/]+)/view[^>]*>.*?<div class="flip-entry-title">([^<]+)</div>', page, re.S)
    return {html.unescape(title).removesuffix(".zip"): file_id for file_id, title in entries}

train_archives, test_archives = folder_files(links["train_folder"]), folder_files(links["test_folder"])
print(len(train_archives), "training archives,", len(test_archives), "test archives")

### Crop

In [ ]:
import tarfile
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm
from face_decision_lab.build import DatasetBuilder, download_ranges, http_range_reader, remote_zip_members, zip_members
from face_decision_lab.faces import RetinaFaceDetector
from face_decision_lab.sources import df40

api = HfApi()
if not Path(BUILD_DIR).exists() and api.repo_exists(WORK_REPO, repo_type = "dataset"):
    with tarfile.open(hf_hub_download(WORK_REPO, "build_df40.tar", repo_type = "dataset")) as tar:
        tar.extractall(Path(BUILD_DIR).parent, filter = "data")
builder = DatasetBuilder(BUILD_DIR, RetinaFaceDetector())
done_dir = Path(BUILD_DIR) / "archives_done"
done_dir.mkdir(exist_ok = True)
Path(ZIP_DIR).mkdir(parents = True, exist_ok = True)
print(len(builder.samples), "crops so far,", len(list(done_dir.iterdir())), "archives done")

def crop_archive(name, file_id, parse, keep, whole = False):
    """Crops the kept members of an archive. whole downloads it first in 64 MB ranges, for archives
    used almost entirely; the others are read member by member."""
    marker = done_dir / f"{name}.done"
    if marker.exists():
        return
    reader = http_range_reader(f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t")
    if whole:
        path = Path(ZIP_DIR) / f"{name}.zip"
        download_ranges(reader, path)
        added = builder.add_images((item, data) for item, data in zip_members(path, parse) if keep(item))
        path.unlink()
    else:
        added = builder.add_images(remote_zip_members(reader, parse, keep))
    builder.flush()
    marker.touch()
    print(f"{name}: {added} images")

for name in tqdm(sorted(df40.KNOWN_METHODS), desc = "known methods"):
    crop_archive(f"train_{name}", train_archives[name], lambda m, a = name: df40.parse_train_member(a, m),
                 lambda item: df40.official_split(item) in ("train", "val") and df40.sampled(item.path, EVERY_FAKE))
crop_archive("ffpp_real", links["ffpp_real"], df40.parse_real_member, lambda item: True, whole = True)
for name in tqdm(sorted(df40.UNKNOWN_METHODS), desc = "unknown methods"):
    crop_archive(f"test_{name}", test_archives[name], lambda m, a = name: df40.parse_unknown_member(a, m),
                 lambda item: not item.subject_id.startswith("ffpp_") or df40.official_split(item) == "test", whole = True)
print(len(builder.samples), "crops")

### Backup

Run it when the crop cell stops before the end (a Drive refusal, a lost runtime): the next runtime restores `BUILD_DIR` from `WORK_REPO` and skips what is done.

In [ ]:
with tarfile.open("/content/build_df40.tar", "w") as tar:
    tar.add(BUILD_DIR, arcname = Path(BUILD_DIR).name)
api.create_repo(WORK_REPO, repo_type = "dataset", private = True, exist_ok = True)
api.upload_file(path_or_fileobj = "/content/build_df40.tar", path_in_repo = "build_df40.tar", repo_id = WORK_REPO, repo_type = "dataset")
print(len(builder.samples), "crops backed up")

### Check

Detection outcome per method, and the share of near-black pixels inside the 1.3x framing (`tighten_crop`) of up to 200 crops per method: black there is padding added where the authors' crop was tighter than the lab's margin, and differs between reals and fakes only if their crops do.

In [ ]:
import numpy as np
from PIL import Image
from face_decision_lab.faces import tighten_crop

for generator in sorted(builder.report.counts):
    print(f"{generator:28s} saved {builder.report.saved_share(generator):6.1%}  {builder.report.counts[generator]}")
by_generator = {}
for sample in builder.samples:
    by_generator.setdefault(sample.generator, []).append(sample)
for generator, part in sorted(by_generator.items()):
    shares = [(np.asarray(tighten_crop(Image.open(Path(BUILD_DIR) / s.image_path).convert("RGB"), 1.3)).max(axis = -1) < 8).mean()
              for s in part[:200]]
    print(f"{generator:28s} crops {len(part):6d}  black inside 1.3x {np.mean(shares):6.2%}")

### Upload

In [ ]:
from face_decision_lab.hub import push_build

print(push_build(BUILD_DIR, HF_DATASET))